In [1]:
import os

In [2]:
%pwd

'c:\\Users\\DELL\\OneDrive\\Desktop\\Kidney-Disease-Classification-Deep-Learning-Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\DELL\\OneDrive\\Desktop\\Kidney-Disease-Classification-Deep-Learning-Project'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    root_dir: Path
    history_path: Path
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    experiment_name: str
    params_image_size: list
    params_batch_size: int

In [6]:
from kidneyDisease.constants import *
from kidneyDisease.utils.common import read_yaml, create_directories, save_json

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_path = CONFIG_FILE_PATH,
        params_path = PARAMS_FILE_PATH            
    ):

        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)
        create_directories([self.config.artifacts_root])


    def get_evaluation_config(self) -> EvaluationConfig:
        evaluation = self.config.evaluation
        training = self.config.training
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, self.config.data_file_name)
        create_directories([
            Path(evaluation.root_dir)
        ])
        
        eval_config = EvaluationConfig(
            root_dir=Path(evaluation.root_dir),
            history_path=Path(training.history_path),
            path_of_model=Path(self.config.training.trained_model_path),
            training_data=Path(training_data), 
            mlflow_uri=MLFLOW_URI,
            experiment_name=MLFLOW_EXPERIMENT_NAME,
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [8]:
import tensorflow as tf
import mlflow
import mlflow.keras
from urllib.parse import urlparse
import os
import json
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

c:\Users\DELL\OneDrive\Desktop\Kidney-Disease-Classification-Deep-Learning-Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split = 0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation='bilinear'
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)


    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()


    def save_score(self) -> None:
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)


    def log_into_mlflow(self):
        # ---------------- MLflow Logging ----------------

        mlflow.set_tracking_uri(self.config.mlflow_uri)
        
        # Create the experiment if it doesn't exist
        mlflow.set_experiment(self.config.experiment_name)
        artifact_dir = self.config.root_dir
        with mlflow.start_run():

            # Log Parameters
            mlflow.log_params(self.config.all_params)

            # Log Metrics
            mlflow.log_metrics({
                "loss": float(self.score[0]),
                "accuracy": float(self.score[1])
            })

            # Log Keras Model
            mlflow.keras.log_model(
                self.model,
                artifact_path="model"
            )

            # -------------------------------------------------
            # Generate Predictions
            # -------------------------------------------------
            self.valid_generator.reset()

            y_true = self.valid_generator.classes

            y_pred = self.model.predict(
                self.valid_generator,
                verbose=1
            )

            y_pred = y_pred.argmax(axis=1)

            class_names = list(self.valid_generator.class_indices.keys())

            # -------------------------------------------------
            # Confusion Matrix
            # -------------------------------------------------
            cm = confusion_matrix(y_true, y_pred)

            disp = ConfusionMatrixDisplay(
                confusion_matrix=cm,
                display_labels=class_names
            )

            disp.plot(cmap="Blues")
            confusion_matrix_path = os.path.join(
                artifact_dir,
                "confusion_matrix.png"
            )
            plt.savefig(confusion_matrix_path)
            plt.close()
            
            mlflow.log_artifact(confusion_matrix_path)

            # -------------------------------------------------
            # Classification Report
            # -------------------------------------------------
            report = classification_report(
                y_true,
                y_pred,
                target_names=class_names
            )
            classification_report_path = os.path.join(
                artifact_dir,
                "classification_report.txt"
            )
            with open(classification_report_path, "w") as f:
                f.write(report)

            mlflow.log_artifact(classification_report_path)

            # -------------------------------------------------
            # Accuracy Curve (if history exists)
            # -------------------------------------------------
            history_path = os.path.join("artifacts", "training", "history.json")

            if os.path.exists(self.config.history_path):

                with open(history_path, "r") as f:
                    history = json.load(f)

                # Accuracy Curve
                plt.figure(figsize=(8,5))
                plt.plot(history["accuracy"], label="Train")
                plt.plot(history["val_accuracy"], label="Validation")
                plt.xlabel("Epoch")
                plt.ylabel("Accuracy")
                plt.legend()

                accuracy_curve_path = os.path.join(
                    artifact_dir,
                    "accuracy_curve.png"
                )
                plt.savefig(accuracy_curve_path)
                plt.close()

                mlflow.log_artifact(accuracy_curve_path)

                # Loss Curve
                plt.figure(figsize=(8,5))
                plt.plot(history["loss"], label="Train")
                plt.plot(history["val_loss"], label="Validation")
                plt.xlabel("Epoch")
                plt.ylabel("Loss")
                plt.legend()

                loss_curve_path = os.path.join(
                    artifact_dir,
                    "loss_curve.png"
                )
                plt.savefig(loss_curve_path)
                plt.close()

                mlflow.log_artifact(loss_curve_path)

                # Log history.json itself
                mlflow.log_artifact(history_path)

            # -------------------------------------------------
            # Log scores.json if present
            # -------------------------------------------------
            if os.path.exists("scores.json"):
                mlflow.log_artifact("scores.json")

        print("✅ MLflow logging completed successfully!")

In [13]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-07-19 19:12:34,357: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-19 19:12:34,358: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-19 19:12:34,361: INFO: common: created directory at: artifacts]
[2026-07-19 19:12:34,363: INFO: common: created directory at: artifacts\evalutation]
[2026-07-19 19:12:34,602: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 122 images belonging to 4 classes.
8/8 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - accuracy: 0.8115 - loss: 2.6227
[2026-07-19 19:12:49,733: INFO: common: json file saved at: scores.json]


2026/07/19 19:12:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/19 19:12:59 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


8/8 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step
🏃 View run big-robin-556 at: https://dagshub.com/Sharif-Abusad/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0/runs/b3beb43d656c46caab3ad257ec8a6fca
🧪 View experiment at: https://dagshub.com/Sharif-Abusad/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0
✅ MLflow logging completed successfully!
